# Workshop de Análise de Dados - Dia 2
## Análise e Comunicação

Ontem você diagnosticou, limpou, explorou e formulou perguntas. Hoje você responde essas perguntas, cria as visualizações finais e monta o documento de análise que a Marina pediu.

Ao final do dia, você terá:
- Respostas quantitativas para cada pergunta
- Visualizações prontas para apresentação
- Um documento de análise completo para seu portfólio

In [ ]:
# Setup + Carregamento de dados
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn
    !mkdir -p data/prepared
    # Tentar carregar arquivos limpos do Dia 1
    # Opção 1: Upload dos arquivos que você baixou ontem
    # Opção 2: Download das versões pré-limpas (fallback)
    BASE_URL = 'https://github.com/caio-gomes/workshop-analise-dados/releases/download/v1/data'
    clean_files = ['orders_clean.csv', 'items_clean.csv', 'reviews_clean.csv']
    
    # Verifica se os arquivos limpos já existem (upload prévio)
    missing_files = [f for f in clean_files if not os.path.exists(f'data/prepared/{f}')]
    
    if missing_files:
        print('Arquivos limpos não encontrados. Escolha uma opção:')
        print('  1) Faça upload dos arquivos que baixou no Dia 1 (recomendado)')
        print('  2) Execute a próxima célula para usar versões pré-limpas')
        print()
        try:
            from google.colab import files
            print('Aguardando upload...')
            uploaded = files.upload()
            for fname in uploaded:
                !mv {fname} data/prepared/{fname}
        except Exception:
            print('Upload cancelado. Baixando versões pré-limpas...')
            for f in clean_files:
                !wget -q -O data/prepared/{f} {BASE_URL}/{f}
    
    DATA_DIR = 'data/prepared'
    print('Ambiente Colab configurado!')
else:
    DATA_DIR = '../data/prepared'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 50)

# Carregue os dados limpos do Dia 1
orders = pd.read_csv(f'{DATA_DIR}/orders_clean.csv', parse_dates=['purchase_date', 'approved_date', 'delivered_date', 'estimated_delivery_date'])
items = pd.read_csv(f'{DATA_DIR}/items_clean.csv')
reviews = pd.read_csv(f'{DATA_DIR}/reviews_clean.csv')

print(f'Orders:  {len(orders):,} linhas')
print(f'Items:   {len(items):,} linhas')
print(f'Reviews: {len(reviews):,} linhas')
print('\nDados limpos carregados!')

In [ ]:
# Crie o dataset consolidado para análise
# Junte orders + items + reviews em um único DataFrame

df = (
    orders
    .merge(items, on='order_id', how='inner')
    .merge(reviews, on='order_id', how='left')
)

# Crie colunas derivadas úteis
df['total_item'] = df['price'] + df['freight_value']
df['delivery_days'] = (df['delivered_date'] - df['purchase_date']).dt.days
df['delay_days'] = (df['delivered_date'] - df['estimated_delivery_date']).dt.days
df['purchase_month'] = df['purchase_date'].dt.to_period('M')

print(f'Dataset consolidado: {len(df):,} linhas, {len(df.columns)} colunas')
display(df.head())

---
# Bloco 1: Análise Estatística Aplicada (60 min)

**Objetivo:** Responder cada pergunta do briefing com dados. Para cada resposta, mostre o número e a interpretação.

### Análise 1: Satisfação por Categoria

Quais categorias têm as piores notas? Qual o volume de cada uma?

In [ ]:
# Calcule nota média, mediana e volume por categoria
cat_stats = (
    df[df['review_score'].notna()]
    .groupby('category')
    .agg(
        nota_media=('review_score', 'mean'),
        nota_mediana=('review_score', 'median'),
        volume=('review_score', 'count'),
        receita=('price', 'sum'),
    )
    .round(2)
)

# Filtre categorias com volume mínimo (evite conclusões de amostras pequenas)
cat_stats_relevant = cat_stats[cat_stats['volume'] >= 100].sort_values('nota_media')

print('Top 10 categorias com PIOR nota média (mínimo 100 avaliações):')
display(cat_stats_relevant.head(10))

print('\nTop 10 categorias com MELHOR nota média:')
display(cat_stats_relevant.tail(10))

In [ ]:
# Sua interpretação:
# O problema de satisfação é generalizado ou concentrado? Quais categorias merecem atenção?
#
# Interpretação: 

### Análise 2: Frete e Satisfação

Atraso na entrega e frete caro afetam a satisfação?

In [ ]:
# Separe pedidos entregues com score válido
delivered = df[
    (df['order_status'] == 'delivered') & 
    (df['review_score'].notna()) &
    (df['delay_days'].notna())
].copy()

# Crie faixas de atraso
delivered['delay_bucket'] = pd.cut(
    delivered['delay_days'],
    bins=[-999, -7, 0, 7, 14, 999],
    labels=['Antecipado (>7d)', 'No prazo (0-7d antes)', 'Leve atraso (0-7d)', 'Atraso (7-14d)', 'Atraso grave (>14d)']
)

# Nota média por faixa de atraso
delay_analysis = delivered.groupby('delay_bucket', observed=True).agg(
    nota_media=('review_score', 'mean'),
    volume=('review_score', 'count')
).round(2)

print('Nota média por faixa de atraso na entrega:')
display(delay_analysis)

In [ ]:
# Análise de frete: divida em quartis de frete e compare nota média
delivered['freight_quartile'] = pd.qcut(
    delivered['freight_value'], 
    q=4, 
    labels=['Q1 (barato)', 'Q2', 'Q3', 'Q4 (caro)']
)

freight_analysis = delivered.groupby('freight_quartile', observed=True).agg(
    nota_media=('review_score', 'mean'),
    frete_medio=('freight_value', 'mean'),
    volume=('review_score', 'count')
).round(2)

print('Nota média por quartil de frete:')
display(freight_analysis)

In [ ]:
# Correlação entre delay_days e review_score
corr = delivered[['delay_days', 'review_score', 'freight_value']].corr()
print('Correlações com review_score:')
print(f'  Atraso vs. Nota:  {corr.loc["delay_days", "review_score"]:.3f}')
print(f'  Frete vs. Nota:   {corr.loc["freight_value", "review_score"]:.3f}')

# Interpretação: qual fator tem mais impacto na satisfação, atraso ou frete?
# Interpretação: 

### Análise 3: Sazonalidade e Tendência

O volume e a satisfação mudam ao longo do tempo?

In [ ]:
# Série temporal: volume e nota média por mês
monthly = (
    df[df['review_score'].notna()]
    .groupby('purchase_month')
    .agg(
        volume=('order_id', 'nunique'),
        nota_media=('review_score', 'mean'),
        receita=('price', 'sum'),
    )
    .round(2)
)

display(monthly)

In [ ]:
# Sazonalidade por dia da semana
df['day_of_week'] = df['purchase_date'].dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dow_stats = (
    df[df['review_score'].notna()]
    .groupby('day_of_week')
    .agg(
        volume=('order_id', 'count'),
        nota_media=('review_score', 'mean'),
    )
    .reindex(dow_order)
    .round(2)
)

print('Volume e nota por dia da semana:')
display(dow_stats)

# Interpretação: há padrão semanal?
# Interpretação: 

### Checkpoint - Bloco 1

In [ ]:
# CHECKPOINT 1 - Análise Estatística
checks = {
    'Dataset consolidado criado': 'df' in dir() and len(df) > 0,
    'Análise por categoria feita': 'cat_stats_relevant' in dir(),
    'Análise de atraso feita': 'delay_analysis' in dir(),
    'Análise mensal feita': 'monthly' in dir(),
}
for check, passed in checks.items():
    status = 'OK' if passed else 'VERIFICAR'
    print(f'  [{status}] {check}')

if all(checks.values()):
    print('\nBloco 1 completo! Siga para as visualizações.')

### Desafio - Bloco 1 (opcional)

A nota média por categoria pode ser enganosa se a distribuição for bimodal (muitos 1s e 5s). Calcule a proporção de notas 1-2 (insatisfeitos) vs. 4-5 (satisfeitos) para as top 10 categorias por volume. Alguma categoria tem distribuição bimodal?

In [ ]:
# Desafio: sua análise aqui


---
# Bloco 2: Visualizações Interpretativas (60 min)

**Objetivo:** Criar 4 visualizações finais. Cada uma responde uma pergunta e tem: título descritivo, eixos rotulados, e anotação que destaca o insight principal. São essas que vão para o documento da Marina.

### Visualização 1: Satisfação por Categoria (Top e Bottom)

In [ ]:
# Visualização 1: Barras horizontais com as 5 melhores e 5 piores categorias
fig, ax = plt.subplots(figsize=(12, 8))

worst5 = cat_stats_relevant.head(5)
best5 = cat_stats_relevant.tail(5)
plot_data = pd.concat([worst5, best5])['nota_media']

colors = ['#e74c3c'] * 5 + ['#2ecc71'] * 5
plot_data.plot(kind='barh', ax=ax, color=colors)

ax.set_title('Categorias com Melhor e Pior Satisfação\n(mínimo 100 avaliações)', fontsize=14)
ax.set_xlabel('Nota Média')
ax.axvline(x=df['review_score'].mean(), color='gray', linestyle='--', alpha=0.5, label=f'Média geral: {df["review_score"].mean():.2f}')
ax.legend()

# Adicione anotação com o insight principal
# Altere o texto para refletir o que VOCÊ encontrou nos dados
ax.annotate(
    'Insight: [descreva o padrão que encontrou]',
    xy=(0.5, -0.08), xycoords='axes fraction',
    fontsize=10, style='italic', color='gray',
    ha='center'
)

plt.tight_layout()
VIZ_DIR = f'{DATA_DIR}/../materiais' if not IN_COLAB else 'materiais'
os.makedirs(VIZ_DIR, exist_ok=True)
plt.savefig(f'{VIZ_DIR}/viz1_satisfacao_categoria.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualização 2: Impacto do Atraso na Satisfação

In [ ]:
# Visualização 2: Nota média por faixa de atraso
fig, ax = plt.subplots(figsize=(12, 6))

delay_plot = delay_analysis.reset_index()
bars = ax.bar(
    range(len(delay_plot)), 
    delay_plot['nota_media'],
    color=['#2ecc71', '#27ae60', '#f39c12', '#e67e22', '#e74c3c'],
    edgecolor='white'
)

# Adicione o volume em cada barra
for i, (bar, vol) in enumerate(zip(bars, delay_plot['volume'])):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'n={vol:,}', ha='center', fontsize=9)

ax.set_xticks(range(len(delay_plot)))
ax.set_xticklabels(delay_plot['delay_bucket'], rotation=15, ha='right')
ax.set_ylabel('Nota Média')
ax.set_title('Quanto Mais Atraso, Pior a Avaliação\nNota média por faixa de atraso na entrega', fontsize=14)
ax.set_ylim(0, 5.5)

# Anotação
ax.annotate(
    'Insight: [descreva o padrão que encontrou]',
    xy=(0.5, -0.15), xycoords='axes fraction',
    fontsize=10, style='italic', color='gray',
    ha='center'
)

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/viz2_atraso_satisfacao.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualização 3: Evolução Temporal

In [ ]:
# Visualização 3: Dual axis - volume e nota ao longo do tempo
fig, ax1 = plt.subplots(figsize=(14, 6))

# Eixo 1: volume
x = range(len(monthly))
ax1.bar(x, monthly['volume'], alpha=0.3, color='steelblue', label='Volume de pedidos')
ax1.set_ylabel('Volume de Pedidos', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Eixo 2: nota média
ax2 = ax1.twinx()
ax2.plot(x, monthly['nota_media'], color='coral', marker='o', linewidth=2, label='Nota média')
ax2.set_ylabel('Nota Média', color='coral')
ax2.tick_params(axis='y', labelcolor='coral')
ax2.set_ylim(1, 5)

# Labels do eixo X
ax1.set_xticks(x)
ax1.set_xticklabels([str(m) for m in monthly.index], rotation=45, ha='right')

ax1.set_title('Volume de Pedidos e Satisfação ao Longo do Tempo', fontsize=14)

# Combine as legendas
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Anotação
fig.text(0.5, -0.02, 'Insight: [descreva o padrão que encontrou]',
         fontsize=10, style='italic', color='gray', ha='center')

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/viz3_evolucao_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualização 4: Mapa de Calor — Onde Categoria e Atraso Se Cruzam

As visualizações anteriores mostraram dois padrões separados: categorias com pior nota e faixas de atraso com pior nota. Agora vamos cruzar os dois numa única imagem.

Um **heatmap** (mapa de calor) usa cores para representar valores numa grade. Cada célula mostra a nota média para uma combinação específica de categoria e faixa de atraso. Vermelho escuro = nota baixa. Verde = nota alta.

In [ ]:
# Visualização 4: Heatmap de satisfação
# Pergunta que responde: Quais categorias sofrem mais com atrasos?

# Selecionar top 8 categorias por volume (evitar ruído de categorias pequenas)
top_cats = df.groupby('category')['order_id'].count().nlargest(8).index
df_top = delivered[delivered['category'].isin(top_cats)]

# Criar tabela cruzada: nota média por categoria × faixa de atraso
pivot = df_top.pivot_table(
    values='review_score',
    index='category',
    columns='delay_bucket',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    pivot,
    annot=True, fmt='.1f',
    cmap='RdYlGn',  # vermelho→amarelo→verde
    center=3.5,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Satisfação por Categoria e Faixa de Atraso', fontsize=14)
ax.set_ylabel('Categoria')
ax.set_xlabel('Faixa de Atraso')

# Insight: [descreva o padrão que você encontrou — quais combinações
#           de categoria + atraso têm as piores notas?]

plt.tight_layout()
plt.savefig(f'{VIZ_DIR}/viz4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Checkpoint - Bloco 2

In [ ]:
# CHECKPOINT 2 - Visualizações
viz_files = [
    f'{VIZ_DIR}/viz1_satisfacao_categoria.png',
    f'{VIZ_DIR}/viz2_atraso_satisfacao.png',
    f'{VIZ_DIR}/viz3_evolucao_temporal.png',
    f'{VIZ_DIR}/viz4_heatmap.png',
]

for f in viz_files:
    exists = os.path.exists(f)
    status = 'OK' if exists else 'FALTA'
    print(f'  [{status}] {os.path.basename(f)}')

all_exist = all(os.path.exists(f) for f in viz_files)
if all_exist:
    print('\n4 visualizações salvas! Siga para o storytelling.')
else:
    print('\nAlgumas visualizações faltam. Complete antes de continuar.')

### Desafio - Bloco 2 (opcional)

Crie um small multiples: o mesmo tipo de gráfico repetido para diferentes segmentos. Exemplo: distribuição de notas para as 6 maiores categorias em um grid 2x3.

In [ ]:
# Desafio: sua visualização aqui


---
# Bloco 3: Storytelling com Dados (45 min)

**Objetivo:** Transformar seus números e gráficos em uma narrativa que a Marina consiga apresentar para a diretoria. A estrutura é: **contexto > problema > descoberta > recomendação**.

## Documento de Análise: Diagnóstico de Satisfação do Marketplace

### Resumo Executivo

_[Escreva 3-5 parágrafos resumindo as principais descobertas e recomendações. Este é o trecho que a diretoria vai ler primeiro. Deve ser compreensível sem ver os gráficos.]_



### Contexto

_[Descreva o cenário: o que motivou esta análise? Qual o problema reportado? Que dados foram usados?]_

- **Período analisado:** [preencha]
- **Volume de pedidos:** [preencha]
- **Nota média geral:** [preencha]



### Descoberta 1: [Título descritivo]

_[Descreva a descoberta. Referencie a Visualização 1.]_

![Satisfação por Categoria](../materiais/viz1_satisfacao_categoria.png)



### Descoberta 2: [Título descritivo]

_[Descreva a descoberta. Referencie a Visualização 2.]_

![Impacto do Atraso](../materiais/viz2_atraso_satisfacao.png)



### Descoberta 3: [Título descritivo]

_[Descreva a descoberta. Referencie a Visualização 3.]_

![Evolução Temporal](../materiais/viz3_evolucao_temporal.png)



### Recomendações

Com base nas descobertas acima, recomendamos:

1. **[Recomendação 1]:** _[descrição e justificativa baseada nos dados]_
2. **[Recomendação 2]:** _[descrição e justificativa baseada nos dados]_

### Limitações e Próximos Passos

_[O que esta análise NÃO responde? O que precisaria de investigação adicional?]_



### Checkpoint - Bloco 3

In [ ]:
# CHECKPOINT 3 - Storytelling
print('Verifique manualmente:')
print('  [ ] Resumo executivo escrito (3-5 parágrafos)?')
print('  [ ] Contexto com período, volume e nota média preenchidos?')
print('  [ ] 3 descobertas com descrição e visualização?')
print('  [ ] Pelo menos 2 recomendações baseadas nos dados?')
print('  [ ] Limitações e próximos passos documentados?')
print('\nSe tudo preenchido, siga para a exportação!')

---
# Bloco 4: Documento Final (45 min)

**Objetivo:** Exportar tudo em um formato apresentável. O resultado final é o que vai para o seu portfólio.

In [ ]:
# Exporte o documento de análise como HTML (sem código)
import subprocess

if IN_COLAB:
    # No Colab, exporta e oferece download
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', '--no-input',
         '--output', 'analise_marketplace',
         '/content/dia2_analise_comunicacao.ipynb'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        from google.colab import files
        print('Documento exportado! Baixando...')
        files.download('analise_marketplace.html')
    else:
        print(f'Erro na exportação: {result.stderr}')
        print('Tente: !jupyter nbconvert --to html --no-input dia2_analise_comunicacao.ipynb')
else:
    output_path = f'{DATA_DIR}/../materiais/analise_marketplace.html'
    result = subprocess.run(
        ['jupyter', 'nbconvert', '--to', 'html', '--no-input',
         '--output', '../materiais/analise_marketplace',
         'dia2_analise_comunicacao.ipynb'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'Documento exportado: {output_path}')
        print('Abra este arquivo no navegador para ver o resultado final.')
    else:
        print('Erro na exportação. Tente manualmente:')
        print('  jupyter nbconvert --to html --no-input dia2_analise_comunicacao.ipynb')
        print(f'  Erro: {result.stderr}')

### Checkpoint Final - Dia 2

In [ ]:
# CHECKPOINT FINAL
print('=== Status Final do Workshop ===')
print()

checks = {
    'Dataset consolidado': 'df' in dir() and len(df) > 0,
    'Análise por categoria': 'cat_stats_relevant' in dir(),
    'Análise de atraso': 'delay_analysis' in dir(),
    'Análise temporal': 'monthly' in dir(),
    'Viz 1 salva': os.path.exists(f'{VIZ_DIR}/viz1_satisfacao_categoria.png'),
    'Viz 2 salva': os.path.exists(f'{VIZ_DIR}/viz2_atraso_satisfacao.png'),
    'Viz 3 salva': os.path.exists(f'{VIZ_DIR}/viz3_evolucao_temporal.png'),
    'Viz 4 salva': os.path.exists(f'{VIZ_DIR}/viz4_heatmap.png'),
}

passed = sum(checks.values())
total = len(checks)

for check, ok in checks.items():
    status = 'OK' if ok else 'FALTA'
    print(f'  [{status}] {check}')

print(f'\nProgresso: {passed}/{total}')

if passed == total:
    print('\nParabéns! Sua análise exploratória está completa.')
    print('Você tem um projeto real para seu portfólio.')
else:
    print(f'\nFaltam {total - passed} itens. Você tem 7 dias para finalizar.')